# Advanced Retrieval with LangChain

In the following notebook, we'll explore various methods of advanced retrieval using LangChain!

We'll touch on:

- Naive Retrieval
- Best-Matching 25 (BM25)
- Multi-Query Retrieval
- Parent-Document Retrieval
- Contextual Compression (a.k.a. Rerank)
- Ensemble Retrieval
- Semantic chunking

We'll also discuss how these methods impact performance on our set of documents with a simple RAG chain.

There will be two breakout rooms:

- 🤝 Breakout Room Part #1
  - Task 1: Getting Dependencies!
  - Task 2: Data Collection and Preparation
  - Task 3: Setting Up QDrant!
  - Task 4-10: Retrieval Strategies
- 🤝 Breakout Room Part #2
  - Activity: Evaluate with Ragas

# 🤝 Breakout Room Part #1

## Task 1: Getting Dependencies!

We're going to need a few specific LangChain community packages, like OpenAI (for our [LLM](https://platform.openai.com/docs/models) and [Embedding Model](https://platform.openai.com/docs/guides/embeddings)) and Cohere (for our [Reranker](https://cohere.com/rerank)).

We'll also provide our OpenAI key, as well as our Cohere API key.

In [53]:
import os
import getpass

os.environ["OPENAI_API_KEY"] = getpass.getpass("Enter your OpenAI API Key:")

In [54]:
os.environ["COHERE_API_KEY"] = getpass.getpass("Cohere API Key:")

## Task 2: Data Collection and Preparation

We'll be using our Use Case Data once again - this time the strutured data available through the CSV!

### Data Preparation

We want to make sure all our documents have the relevant metadata for the various retrieval strategies we're going to be applying today.

In [55]:
from langchain_community.document_loaders import DirectoryLoader, PyMuPDFLoader
from pathlib import Path

DATA_DIR = Path("data/grade3")
assert DATA_DIR.exists(), f"Missing dataset folder: {DATA_DIR.resolve()}"

loader = DirectoryLoader(str(DATA_DIR), glob="*.pdf", loader_cls=PyMuPDFLoader)
docs = loader.load()

# Keep variable name consistent with rest of notebook
synthetic_usecase_data = docs

for doc in synthetic_usecase_data:
    doc.metadata.setdefault("source", doc.metadata.get("source", "kids_science_pdf"))

Let's look at an example document to see if everything worked as expected!

In [56]:
synthetic_usecase_data[0]

Document(metadata={'producer': 'WeasyPrint 65.1', 'creator': 'ChatGPT', 'creationdate': '', 'source': 'data/grade3/Grade 3 Science Booklets (Ontario Curriculum Aligned).pdf', 'file_path': 'data/grade3/Grade 3 Science Booklets (Ontario Curriculum Aligned).pdf', 'total_pages': 25, 'format': 'PDF 1.7', 'title': 'Grade 3 Science Booklets (Ontario Curriculum Aligned)', 'author': 'ChatGPT Deep Research', 'subject': '', 'keywords': '', 'moddate': '', 'trapped': '', 'modDate': '', 'creationDate': '', 'page': 0}, page_content='Grade 3 Science Booklets (Ontario Curriculum\nAligned)\nBelow are outlines for five Grade 3 science booklets, each aligned with the Ontario Science & Technology\ncurriculum  (TVO  Learn).  Four  booklets  cover  the  major  science  strands  (Life  Systems,  Structures  &\nMechanisms, Matter & Energy, Earth & Space Systems), and the fifth booklet is a standalone “Just the Facts”\nreference. Each booklet is designed to be print-friendly (A4, black-and-white) and includes a

## Task 3: Setting up QDrant!

Now that we have our documents, let's create a QDrant VectorStore with the collection name "Synthetic_Usecases".

We'll leverage OpenAI's [`text-embedding-3-small`](https://openai.com/blog/new-embedding-models-and-api-updates) because it's a very powerful (and low-cost) embedding model.

> NOTE: We'll be creating additional vectorstores where necessary, but this pattern is still extremely useful.

In [57]:
from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vectorstore = Qdrant.from_documents(
    synthetic_usecase_data,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecases"
)

In [58]:
# === Eval-only helpers (do not collide with Task 8) ===
# TODO(you): eval-only; do not override Task 8 variables.
from langchain_text_splitters import RecursiveCharacterTextSplitter
splitter_eval = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)  # TODO(you): tweak if needed
parent_docs_full = synthetic_usecase_data
child_docs_eval = splitter_eval.split_documents(parent_docs_full)
print(f"[Eval helpers] parents={len(parent_docs_full)} | child_chunks={len(child_docs_eval)}")


[Eval helpers] parents=25 | child_chunks=186


## Task 4: Naive RAG Chain

Since we're focusing on the "R" in RAG today - we'll create our Retriever first.

### R - Retrieval

This naive retriever will simply look at each review as a document, and use cosine-similarity to fetch the 10 most relevant documents.

> NOTE: We're choosing `10` as our `k` here to provide enough documents for our reranking process later

In [7]:
naive_retriever = vectorstore.as_retriever(search_kwargs={"k" : 10})
base_retriever = naive_retriever  # TODO(you): shared handle for later/eval cells

### A - Augmented

We're going to go with a standard prompt for our simple RAG chain today! Nothing fancy here, we want this to mostly be about the Retrieval process.

In [8]:
from langchain_core.prompts import ChatPromptTemplate

RAG_TEMPLATE = """\
You are a helpful and kind assistant. Use the context provided below to answer the question.

If you do not know the answer, or are unsure, say you don't know.

Query:
{question}

Context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_template(RAG_TEMPLATE)

### G - Generation

We're going to leverage `gpt-4.1-nano` as our LLM today, as - again - we want this to largely be about the Retrieval process.

In [9]:
from langchain_openai import ChatOpenAI

chat_model = ChatOpenAI(model="gpt-4.1-nano")

### LCEL RAG Chain

We're going to use LCEL to construct our chain.

> NOTE: This chain will be exactly the same across the various examples with the exception of our Retriever!

In [10]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
from langchain_core.runnables import RunnablePassthrough
from operator import itemgetter
from langchain_core.output_parsers import StrOutputParser

naive_retrieval_chain = (
    # INVOKE CHAIN WITH: {"question" : "<<SOME USER QUESTION>>"}
    # "question" : populated by getting the value of the "question" key
    # "context"  : populated by getting the value of the "question" key and chaining it into the base_retriever
    {"context": itemgetter("question") | naive_retriever, "question": itemgetter("question")}
    # "context"  : is assigned to a RunnablePassthrough object (will not be called or considered in the next step)
    #              by getting the value of the "context" key from the previous step
    | RunnablePassthrough.assign(context=itemgetter("context"))
    # "response" : the "context" and "question" values are used to format our prompt object and then piped
    #              into the LLM and stored in a key called "response"
    # "context"  : populated by getting the value of the "context" key from the previous step
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's see how this simple chain does on a few different prompts.

> NOTE: You might think that we've cherry picked prompts that showcase the individual skill of each of the retrieval strategies - you'd be correct!

#TODO change these questions to more science related

In [11]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
#naive_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

In [12]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
#naive_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

In [13]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
#naive_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

Overall, this is not bad! Let's see if we can make it better!

## Task 5: Best-Matching 25 (BM25) Retriever

Taking a step back in time - [BM25](https://www.nowpublishers.com/article/Details/INR-019) is based on [Bag-Of-Words](https://en.wikipedia.org/wiki/Bag-of-words_model) which is a sparse representation of text.

In essence, it's a way to compare how similar two pieces of text are based on the words they both contain.

This retriever is very straightforward to set-up! Let's see it happen down below!


In [14]:
from langchain_community.retrievers import BM25Retriever

bm25_retriever = BM25Retriever.from_documents(synthetic_usecase_data)

We'll construct the same chain - only changing the retriever.

In [15]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
bm25_retrieval_chain = (
    {"context": itemgetter("question") | bm25_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at the responses!

In [16]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
#bm25_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

In [17]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
#bm25_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

In [18]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
#bm25_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

It's not clear that this is better or worse, if only we had a way to test this (SPOILERS: We do, the second half of the notebook will cover this)

#### ❓ Question #1:

Give an example query where BM25 is better than embeddings and justify your answer.

##### ✅ Answer


## Task 6: Contextual Compression (Using Reranking)

Contextual Compression is a fairly straightforward idea: We want to "compress" our retrieved context into just the most useful bits.

There are a few ways we can achieve this - but we're going to look at a specific example called reranking.

The basic idea here is this:

- We retrieve lots of documents that are very likely related to our query vector
- We "compress" those documents into a smaller set of *more* related documents using a reranking algorithm.

We'll be leveraging Cohere's Rerank model for our reranker today!

All we need to do is the following:

- Create a basic retriever
- Create a compressor (reranker, in this case)

That's it!

Let's see it in the code below!

In [19]:
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank

compressor = CohereRerank(model="rerank-v3.5")
compression_retriever = ContextualCompressionRetriever(
    base_compressor=compressor, base_retriever=naive_retriever
)

Let's create our chain again, and see how this does!

In [20]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
contextual_compression_retrieval_chain = (
    {"context": itemgetter("question") | compression_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [21]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
#contextual_compression_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

In [22]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
#contextual_compression_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

In [23]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
#contextual_compression_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

We'll need to rely on something like Ragas to help us get a better sense of how this is performing overall - but it "feels" better!

## Task 7: Multi-Query Retriever

Typically in RAG we have a single query - the one provided by the user.

What if we had....more than one query!

In essence, a Multi-Query Retriever works by:

1. Taking the original user query and creating `n` number of new user queries using an LLM.
2. Retrieving documents for each query.
3. Using all unique retrieved documents as context

So, how is it to set-up? Not bad! Let's see it down below!



In [24]:
from langchain.retrievers.multi_query import MultiQueryRetriever

multi_query_retriever = MultiQueryRetriever.from_llm(
    retriever=naive_retriever, llm=chat_model
) 

In [25]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
multi_query_retrieval_chain = (
    {"context": itemgetter("question") | multi_query_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

In [26]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
#multi_query_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

In [27]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
#multi_query_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

In [28]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
#multi_query_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

#### ❓ Question #2:

Explain how generating multiple reformulations of a user query can improve recall.

##### ✅ Answer


## Task 8: Parent Document Retriever

A "small-to-big" strategy - the Parent Document Retriever works based on a simple strategy:

1. Each un-split "document" will be designated as a "parent document" (You could use larger chunks of document as well, but our data format allows us to consider the overall document as the parent chunk)
2. Store those "parent documents" in a memory store (not a VectorStore)
3. We will chunk each of those documents into smaller documents, and associate them with their respective parents, and store those in a VectorStore. We'll call those "child chunks".
4. When we query our Retriever, we will do a similarity search comparing our query vector to the "child chunks".
5. Instead of returning the "child chunks", we'll return their associated "parent chunks".

Okay, maybe that was a few steps - but the basic idea is this:

- Search for small documents
- Return big documents

The intuition is that we're likely to find the most relevant information by limiting the amount of semantic information that is encoded in each embedding vector - but we're likely to miss relevant surrounding context if we only use that information.

Let's start by creating our "parent documents" and defining a `RecursiveCharacterTextSplitter`.

In [29]:
from langchain.retrievers import ParentDocumentRetriever
from langchain.storage import InMemoryStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from qdrant_client import QdrantClient, models

parent_docs = synthetic_usecase_data
child_splitter = RecursiveCharacterTextSplitter(chunk_size=750)

We'll need to set up a new QDrant vectorstore - and we'll use another useful pattern to do so!

> NOTE: We are manually defining our embedding dimension, you'll need to change this if you're using a different embedding model.

In [30]:
from langchain_qdrant import QdrantVectorStore

client = QdrantClient(location=":memory:")

client.create_collection(
    collection_name="full_documents",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE)
)

parent_document_vectorstore = QdrantVectorStore(
    collection_name="full_documents", embedding=OpenAIEmbeddings(model="text-embedding-3-small"), client=client
)

Now we can create our `InMemoryStore` that will hold our "parent documents" - and build our retriever!

In [31]:
store = InMemoryStore()

parent_document_retriever = ParentDocumentRetriever(
    vectorstore = parent_document_vectorstore,
    docstore=store,
    child_splitter=child_splitter,
)

By default, this is empty as we haven't added any documents - let's add some now!

In [32]:
parent_document_retriever.add_documents(parent_docs, ids=None)

We'll create the same chain we did before - but substitute our new `parent_document_retriever`.

In [33]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
parent_document_retrieval_chain = (
    {"context": itemgetter("question") | parent_document_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's give it a whirl!

In [34]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
#parent_document_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

In [35]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
#parent_document_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

In [36]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
#parent_document_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

Overall, the performance *seems* largely the same. We can leverage a tool like [Ragas]() to more effectively answer the question about the performance.

## Task 9: Ensemble Retriever

In brief, an Ensemble Retriever simply takes 2, or more, retrievers and combines their retrieved documents based on a rank-fusion algorithm.

In this case - we're using the [Reciprocal Rank Fusion](https://plg.uwaterloo.ca/~gvcormac/cormacksigir09-rrf.pdf) algorithm.

Setting it up is as easy as providing a list of our desired retrievers - and the weights for each retriever.

In [37]:
from langchain.retrievers import EnsembleRetriever

retriever_list = [bm25_retriever, naive_retriever, parent_document_retriever, compression_retriever, multi_query_retriever]
equal_weighting = [1/len(retriever_list)] * len(retriever_list)

ensemble_retriever = EnsembleRetriever(
    retrievers=retriever_list, weights=equal_weighting
)

We'll pack *all* of these retrievers together in an ensemble.

In [38]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
ensemble_retrieval_chain = (
    {"context": itemgetter("question") | ensemble_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

Let's look at our results!

In [39]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
#ensemble_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

In [40]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
#ensemble_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

In [41]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
#ensemble_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

## Task 10: Semantic Chunking

While this is not a retrieval method - it *is* an effective way of increasing retrieval performance on corpora that have clean semantic breaks in them.

Essentially, Semantic Chunking is implemented by:

1. Embedding all sentences in the corpus.
2. Combining or splitting sequences of sentences based on their semantic similarity based on a number of [possible thresholding methods](https://python.langchain.com/docs/how_to/semantic-chunker/):
  - `percentile`
  - `standard_deviation`
  - `interquartile`
  - `gradient`
3. Each sequence of related sentences is kept as a document!

Let's see how to implement this!

We'll use the `percentile` thresholding method for this example which will:

Calculate all distances between sentences, and then break apart sequences of setences that exceed a given percentile among all distances.

In [42]:
from langchain_experimental.text_splitter import SemanticChunker

semantic_chunker = SemanticChunker(
    embeddings,
    breakpoint_threshold_type="percentile"
)

Now we can split our documents.

In [43]:
semantic_documents = semantic_chunker.split_documents(synthetic_usecase_data[:20])

Let's create a new vector store.

In [44]:
semantic_vectorstore = Qdrant.from_documents(
    semantic_documents,
    embeddings,
    location=":memory:",
    collection_name="Synthetic_Usecase_Data_Semantic_Chunks"
)

We'll use naive retrieval for this example.

In [45]:
semantic_retriever = semantic_vectorstore.as_retriever(search_kwargs={"k" : 10})

Finally we can create our classic chain!

In [46]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
semantic_retrieval_chain = (
    {"context": itemgetter("question") | semantic_retriever, "question": itemgetter("question")}
    | RunnablePassthrough.assign(context=itemgetter("context"))
    | {"response": rag_prompt | chat_model, "context": itemgetter("context")}
)

And view the results!

In [47]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
#semantic_retrieval_chain.invoke({"question" : "What is the most common project domain?"})["response"].content

In [48]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
#semantic_retrieval_chain.invoke({"question" : "Were there any usecases about security?"})["response"].content

In [49]:
# ⚠️ NOTE: This demo cell uses full PDF pages and may be SLOW. For evaluation, skip to Activity #1.
#semantic_retrieval_chain.invoke({"question" : "What did judges have to say about the fintech projects?"})["response"].content

#### ❓ Question #3:

If sentences are short and highly repetitive (e.g., FAQs), how might semantic chunking behave, and how would you adjust the algorithm?

##### ✅ Answer


# 🤝 Breakout Room Part #2

#### 🏗️ Activity #1

Your task is to evaluate the various Retriever methods against eachother.

You are expected to:

1. Create a "golden dataset"
 - Use Synthetic Data Generation (powered by Ragas, or otherwise) to create this dataset
2. Evaluate each retriever with *retriever specific* Ragas metrics
 - Semantic Chunking is not considered a retriever method and will not be required for marks, but you may find it useful to do a "semantic chunking on" vs. "semantic chunking off" comparision between them
3. Compile these in a list and write a small paragraph about which is best for this particular data and why.

Your analysis should factor in:
  - Cost
  - Latency
  - Performance

> NOTE: This is **NOT** required to be completed in class. Please spend time in your breakout rooms creating a plan before moving on to writing code.

##### HINTS:

- LangSmith provides detailed information about latency and cost.

In [50]:
### YOUR CODE HERE

In [61]:
# === Activity #1 Solution (Eval-only, chunk-based) ===
# TODO(you): This section is for evaluation only; students should not run demo tasks here.

import time
time.sleep(15)  # Wait to avoid Cohere rate limit from previous cells


from langchain_community.vectorstores import Qdrant
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_community.retrievers import BM25Retriever
from langchain.retrievers.contextual_compression import ContextualCompressionRetriever
from langchain_cohere import CohereRerank
from langchain.retrievers.multi_query import MultiQueryRetriever
from langchain.retrievers import ParentDocumentRetriever, EnsembleRetriever
from langchain.storage import InMemoryStore
from langchain_qdrant import QdrantVectorStore
from qdrant_client import QdrantClient, models

EMBED_MODEL = "text-embedding-3-small"
K = 10

embeddings = OpenAIEmbeddings(model=EMBED_MODEL)

# Chunked eval vectorstore (separate from demo)
child_vs_eval = Qdrant.from_documents(child_docs_eval, embeddings, location=":memory:", collection_name="Kids_Science_Eval")
naive_eval = child_vs_eval.as_retriever(search_kwargs={"k": K})
bm25_eval = BM25Retriever.from_documents(child_docs_eval)

# Rerank guardrail
use_rerank = bool(os.environ.get("COHERE_API_KEY"))
if use_rerank:
    compressor = CohereRerank(model="rerank-v3.5")
    compression_eval = ContextualCompressionRetriever(base_compressor=compressor, base_retriever=naive_eval)
else:
    compression_eval = naive_eval  # TODO(you): fallback if rerank key missing
    print("⚠️ Skipping Cohere Rerank (no COHERE_API_KEY). Using naive_eval as fallback.")

multi_query_eval = MultiQueryRetriever.from_llm(retriever=naive_eval, llm=ChatOpenAI(model="gpt-4.1-nano"))

# Parent-Doc (separate collection)
parent_client = QdrantClient(location=":memory:")
parent_client.create_collection(
    collection_name="Kids_Science_Parents_Eval",
    vectors_config=models.VectorParams(size=1536, distance=models.Distance.COSINE),
)
parent_vs_eval = QdrantVectorStore(collection_name="Kids_Science_Parents_Eval", embedding=embeddings, client=parent_client)
parent_store = InMemoryStore()
parent_doc_eval = ParentDocumentRetriever(vectorstore=parent_vs_eval, docstore=parent_store, child_splitter=splitter_eval)
parent_doc_eval.add_documents(parent_docs_full, ids=None)

retriever_list_eval = [bm25_eval, naive_eval, compression_eval, multi_query_eval, parent_doc_eval]
ensemble_eval = EnsembleRetriever(retrievers=retriever_list_eval, weights=[1/len(retriever_list_eval)]*len(retriever_list_eval))

print("✅ Eval retrievers ready (chunked). K=10, chunk=500/50")


✅ Eval retrievers ready (chunked). K=10, chunk=500/50


In [62]:
# --- SDG: golden dataset ---
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from ragas.testset import TestsetGenerator
generator_llm = LangchainLLMWrapper(ChatOpenAI(model="gpt-4.1-nano"))
generator_emb = LangchainEmbeddingsWrapper(embeddings)
generator = TestsetGenerator(llm=generator_llm, embedding_model=generator_emb)
# Use generate() instead of generate_with_langchain_docs() to support num_personas
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset.transforms import default_transforms, apply_transforms

# Build knowledge graph
kg = KnowledgeGraph()
for doc in parent_docs_full:
    kg.nodes.append(Node(type=NodeType.DOCUMENT, properties={"page_content": doc.page_content, "document_metadata": doc.metadata}))

# Apply transforms (with delay to avoid rate limits)
import time
time.sleep(10)  # Wait to avoid Cohere rate limit
transforms = default_transforms(documents=parent_docs_full, llm=generator_llm, embedding_model=generator_emb)
apply_transforms(kg, transforms)

# Generate with limited personas
generator.knowledge_graph = kg
testset = generator.generate(testset_size=10, num_personas=1, raise_exceptions=False)
parent_docs_full,
testset_size=10,
num_personas=1  # Limit to 1 persona to avoid lookup bugs)
test_df = testset.to_pandas()
print("Testset preview shown below (eval-only):")
display(test_df.head())

# --- RAGAS evaluation ---
import time, pandas as pd
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from operator import itemgetter
from ragas.evaluation import evaluate
from ragas.metrics import faithfulness, context_precision, context_recall

def format_docs(docs): return "\n\n".join(d.page_content for d in docs)
prompt = ChatPromptTemplate.from_template(
    "Use ONLY the context to answer. If unsure, say 'I don't know'.\n\nContext:\n{context}\n\nQuestion:\n{question}"
)
llm = ChatOpenAI(model="gpt-4.1-nano")
def make_chain(r):
    return (
        {"question": itemgetter("question")}
        | {"context": itemgetter("question") | r}
        | RunnablePassthrough.assign(context=lambda x: format_docs(x["context"]))
        | prompt | llm | StrOutputParser()
    )

targets = {
    "Naive": naive_eval,
    "BM25": bm25_eval,
    "Rerank": compression_eval,
    "MultiQuery": multi_query_eval,
    "ParentDoc": parent_doc_eval,
    "Ensemble": ensemble_eval,
}

def get_contexts(retr, questions, k=10):
    ctxs = []
    for q in questions:
        docs = retr.get_relevant_documents(q)
        ctxs.append([d.page_content for d in docs])
    return ctxs

def build_ragas_ds(questions, contexts, ground_truth, answers=None):
    data = {"question": questions, "contexts": contexts, "ground_truth": ground_truth}
    if answers is not None:
        data["answer"] = answers
    from datasets import Dataset as HFDataset
    return HFDataset.from_dict(data)

def answer_with_chain(chain, questions):
    return [chain.invoke({"question": q}) for q in questions]

import random, numpy as np
random.seed(42); np.random.seed(42)

EVAL_WITH_FAITHFULNESS = True  # set False to skip answer generation (cheaper/faster)
metrics = [context_precision, context_recall] + ([faithfulness] if EVAL_WITH_FAITHFULNESS else [])

rows = []
for name, retr in targets.items():
    time.sleep(7)  # Cohere rate limit: 10/min
    print(f"\n🔎 Evaluating {name} ...")
    t0 = time.time()

    questions = test_df["user_input"].tolist()
    ground_truth = test_df["reference"].tolist()  # ✅ Fixed: string ground-truth, not list

    contexts = get_contexts(retr, questions, k=10)

    if EVAL_WITH_FAITHFULNESS:
        chain = make_chain(retr)
        answers = answer_with_chain(chain, questions)
        ds = build_ragas_ds(questions, contexts, ground_truth, answers)
    else:
        ds = build_ragas_ds(questions, contexts, ground_truth)

    res = evaluate(dataset=ds, metrics=metrics)

    rows.append({
        "Retriever": name,
        "ContextPrecision": float(res["context_precision"]) if isinstance(res["context_precision"], (int, float)) else float(sum(res["context_precision"]) / len(res["context_precision"])),
        "ContextRecall": float(res["context_recall"]) if isinstance(res["context_recall"], (int, float)) else float(sum(res["context_recall"]) / len(res["context_recall"])),
        "Faithfulness": float(res["faithfulness"]) if isinstance(res["faithfulness"], (int, float)) else float(sum(res["faithfulness"]) / len(res["faithfulness"])) if EVAL_WITH_FAITHFULNESS else None,
        "Latency(s)": round(time.time() - t0, 2),
    })

eval_df = pd.DataFrame(rows).sort_values(
    ["Faithfulness" if EVAL_WITH_FAITHFULNESS else "ContextRecall", "ContextPrecision"],
    ascending=False
).reset_index(drop=True)

print(f"\n📊 Activity #1 — Evaluation (Eval-only, chunked) | k=10 | chunk=500/50 | rerank={'ON' if bool(os.environ.get('COHERE_API_KEY')) else 'OFF'}")
display(eval_df)

# Sanity check on top retriever
top = eval_df.iloc[0]["Retriever"]
q = test_df.sample(1, random_state=42).iloc[0]["user_input"]
print(f"\nTop: {top}\nQ: {q}\n")
print(make_chain(targets[top]).invoke({"question": q}))

print("\n" + "="*60)
print("📊 View traces & costs in LangSmith:")
print("https://smith.langchain.com/")


Applying HeadlinesExtractor:   0%|          | 0/24 [00:00<?, ?it/s]

Applying HeadlineSplitter:   0%|          | 0/25 [00:00<?, ?it/s]

unable to apply transformation: 'headlines' property not found in this node


Applying SummaryExtractor:   0%|          | 0/47 [00:00<?, ?it/s]

Property 'summary' already exists in node '3ad2b9'. Skipping!
Property 'summary' already exists in node '13c516'. Skipping!
Property 'summary' already exists in node 'e62416'. Skipping!
Property 'summary' already exists in node '6e5f0c'. Skipping!
Property 'summary' already exists in node '5857dd'. Skipping!
Property 'summary' already exists in node '2bb74f'. Skipping!
Property 'summary' already exists in node '075794'. Skipping!
Property 'summary' already exists in node '1bdb2c'. Skipping!
Property 'summary' already exists in node 'e51d54'. Skipping!
Property 'summary' already exists in node '6543ef'. Skipping!
Property 'summary' already exists in node '869d31'. Skipping!
Property 'summary' already exists in node '7eb800'. Skipping!
Property 'summary' already exists in node 'ea6668'. Skipping!
Property 'summary' already exists in node '9ffe75'. Skipping!
Property 'summary' already exists in node 'c214e9'. Skipping!
Property 'summary' already exists in node 'ee3fdd'. Skipping!
Property

...


🔎 Evaluating Ensemble ...


Evaluating:   0%|          | 0/15 [00:00<?, ?it/s]


📊 Activity #1 — Evaluation (Eval-only, chunked) | k=10 | chunk=500/50 | rerank=ON


,Retriever,ContextPrecision,ContextRecall,Faithfulness,Latency(s)
0,ParentDoc,1.000000,0.80,1.000000,33.80
1,MultiQuery,0.769602,0.80,1.000000,47.84
2,Naive,0.850668,0.80,0.966667,44.30
3,BM25,0.611111,0.45,0.950000,24.59
4,Ensemble,0.768805,0.80,0.942510,75.14
5,Rerank,1.000000,0.75,0.873333,38.80



Top: ParentDoc
Q: How does humus contribute to the characteristics of loam soil, and why is it important for plant growth?

Humus contributes to loam soil by enriching it with organic matter, making the soil dark, crumbly, and rich in nutrients. It helps the soil hold water and supplies essential nutrients that plants need to grow. This makes loam soil ideal for plant growth because it retains water and nutrients effectively while also draining well, providing a balanced environment for roots.

📊 View traces & costs in LangSmith:
https://smith.langchain.com/


In [63]:
# Sanity check the built dataset shape for one retriever (no faithfulness)
EVAL_WITH_FAITHFULNESS = False
metrics = [context_precision, context_recall]

qs = test_df["user_input"].tolist()[:3]
gt = test_df["reference"].tolist()[:3]  # string ground-truth answers
ctxs = get_contexts(naive_eval, qs, k=10)

ds = build_ragas_ds(qs, ctxs, gt)
print(ds)  # should show columns: question, contexts(list[str]), ground_truth(str)

print("Running a tiny eval…")
tiny_res = evaluate(dataset=ds, metrics=metrics)
print(tiny_res)


Dataset({
    features: ['question', 'contexts', 'ground_truth'],
    num_rows: 3
})
Running a tiny eval…


Evaluating:   0%|          | 0/6 [00:00<?, ?it/s]

{'context_precision': 0.8565, 'context_recall': 1.0000}


In [65]:
import os
import getpass

# Set LangSmith tracing
os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_PROJECT"] = "s9-advanced-retrieval"

# Prompt for LangChain API key if not set
if not os.environ.get("LANGCHAIN_API_KEY"):
    os.environ["LANGCHAIN_API_KEY"] = getpass.getpass("LangChain API Key:")
else:
    print("✅ LANGCHAIN_API_KEY already set")


✅ LANGCHAIN_API_KEY already set


In [66]:
# === LangSmith: create dataset from SDG test_df (minimal, no changes to RAGAS) ===
from langsmith import Client

DATASET_NAME = "Kids Science S9 — SDG"
client = Client()

# Create or reuse dataset
try:
    ls_dataset = client.create_dataset(
        dataset_name=DATASET_NAME,
        description="Kids Science synthetic Q/A/contexts from RAGAS testset"
    )
except Exception:
    ds = [d for d in client.list_datasets() if d.name == DATASET_NAME]
    ls_dataset = ds[0] if ds else client.create_dataset(
        dataset_name=DATASET_NAME,
        description="Kids Science synthetic Q/A/contexts from RAGAS testset"
    )

# Clear old examples to avoid duplicates
try:
    for ex in client.list_examples(dataset_id=ls_dataset.id):
        client.delete_example(ex.id)
    print("🗑️  Cleared old examples")
except Exception:
    pass

# Push examples
for _, r in test_df.iterrows():
    client.create_example(
        inputs={"question": r["user_input"]},
        outputs={"answer": r["reference"]},
        metadata={"context": r["reference_contexts"]},
        dataset_id=ls_dataset.id,
    )

print(f"✅ LangSmith dataset ready: {DATASET_NAME}")


🗑️  Cleared old examples
✅ LangSmith dataset ready: Kids Science S9 — SDG


In [67]:
# === LangSmith: run evaluators for each retriever (keeps RAGAS flow untouched) ===
from langsmith.evaluation import LangChainStringEvaluator, evaluate
from langchain_openai import ChatOpenAI
import time

# Evaluators
eval_llm = ChatOpenAI(model="gpt-4.1")

qa = LangChainStringEvaluator("qa", config={"llm": eval_llm})

helpful = LangChainStringEvaluator(
    "labeled_criteria",
    config={
        "criteria": {"helpfulness": "Is this submission helpful given the reference answer?"},
        "llm": eval_llm,
    },
    prepare_data=lambda run, ex: {
        "prediction": run.outputs["output"],
        "reference": ex.outputs["answer"],
        "input": ex.inputs["question"],
    },
)

dopeness = LangChainStringEvaluator(
    "criteria",
    config={
        "criteria": {"dopeness": "Is the response engaging (non-generic) while remaining grounded?"},
        "llm": eval_llm,
    },
)

# Use existing `targets` (retriever name -> retriever) and `make_chain(retriever)`
for name, retr in targets.items():
    time.sleep(7)  # Cohere Trial: 10/min → need 6s+ delay
    print(f"\n🚀 LangSmith evaluate — {name}")
    chain = make_chain(retr)
    evaluate(
        chain.invoke,
        data=DATASET_NAME,
        evaluators=[qa, helpful, dopeness],
        metadata={"revision_id": f"s9_{name}"},
    )

print("\n📎 In LangSmith, open Datasets → "
      f"'{DATASET_NAME}' and Compare runs by revision_id (s9_<retriever-name>).")



🚀 LangSmith evaluate — Naive
View the evaluation results for experiment: 'crushing-boy-95' at:
https://smith.langchain.com/o/3cf98b72-eeb5-42cf-8a48-12ca0dadba6b/datasets/edeb9e37-4b1d-4fc1-80dd-363a3cbe625f/compare?selectedSessions=3714e603-dca1-46ec-8dea-32229fe1e224




0it [00:00, ?it/s]


🚀 LangSmith evaluate — BM25
View the evaluation results for experiment: 'helpful-kitten-29' at:
https://smith.langchain.com/o/3cf98b72-eeb5-42cf-8a48-12ca0dadba6b/datasets/edeb9e37-4b1d-4fc1-80dd-363a3cbe625f/compare?selectedSessions=3a657e68-a3f8-4cce-8a35-4739256a0301




0it [00:00, ?it/s]


🚀 LangSmith evaluate — Rerank
View the evaluation results for experiment: 'definite-current-87' at:
https://smith.langchain.com/o/3cf98b72-eeb5-42cf-8a48-12ca0dadba6b/datasets/edeb9e37-4b1d-4fc1-80dd-363a3cbe625f/compare?selectedSessions=e8ffca6f-0e96-4d3e-adf7-7a72b72931a2




...


🚀 LangSmith evaluate — ParentDoc
View the evaluation results for experiment: 'reflecting-cabbage-4' at:
https://smith.langchain.com/o/3cf98b72-eeb5-42cf-8a48-12ca0dadba6b/datasets/edeb9e37-4b1d-4fc1-80dd-363a3cbe625f/compare?selectedSessions=0acd6adf-1e56-4834-89ef-eba40945a95e




0it [00:00, ?it/s]


🚀 LangSmith evaluate — Ensemble
View the evaluation results for experiment: 'complicated-hour-30' at:
https://smith.langchain.com/o/3cf98b72-eeb5-42cf-8a48-12ca0dadba6b/datasets/edeb9e37-4b1d-4fc1-80dd-363a3cbe625f/compare?selectedSessions=7df32103-472f-495d-a240-7388799e1944




0it [00:00, ?it/s]


📎 In LangSmith, open Datasets → 'Kids Science S9 — SDG' and Compare runs by revision_id (s9_<retriever-name>).


In [68]:
# === Quick compare reminder (no data mutation) ===
print("✅ You now have BOTH:")
print("  • RAGAS table (retriever-centric: context_precision/recall/faithfulness)")
print("  • LangSmith runs (end-to-end answer quality + latency/cost; revision_id = s9_<retriever>)")
print("\nTip: If rankings disagree, inspect prompts/answers in LangSmith and retrieved contexts.")
print("\n📊 View LangSmith results:")
print(f"https://smith.langchain.com/ → Datasets → '{DATASET_NAME}' → Compare")


✅ You now have BOTH:
  • RAGAS table (retriever-centric: context_precision/recall/faithfulness)
  • LangSmith runs (end-to-end answer quality + latency/cost; revision_id = s9_<retriever>)

Tip: If rankings disagree, inspect prompts/answers in LangSmith and retrieved contexts.

📊 View LangSmith results:
https://smith.langchain.com/ → Datasets → 'Kids Science S9 — SDG' → Compare
